In [0]:
# Versions of Databricks code are not locked since Databricks ensures changes are backwards compatible.
%pip install -qqqq -U databricks-agents databricks-vectorsearch databricks-sdk mlflow mlflow-skinny databricks-langchain
# Restart to load the packages into the Python environment
dbutils.library.restartPython()

In [0]:
%run ./00_config


## Local retriever tool

The following code prototypes a retriever tool and binds it to an LLM locally so you can chat with the agent to test its tool-calling behavior.

In [0]:
from databricks_langchain import VectorSearchRetrieverTool, ChatDatabricks

# Initialize the retriever tool
vs_tool = VectorSearchRetrieverTool(index_name=f"{UC_CATALOG}.{UC_SCHEMA}.wikipedia_vector_index")

# Run a query against the vector search index locally for testing
vs_tool.invoke("Jack climbs a beanstalk")



In [0]:
# Bind the retriever tool to your Langchain LLM of choice
llm = ChatDatabricks(endpoint=CHAT_MODEL_NAME)
llm_with_tools = llm.bind_tools([vs_tool])

# Chat with your LLM to test the tool calling functionality
llm_with_tools.invoke("In which movie does jack climb a beanstalk?")


### Configuring more advanced options for your retriever

In [0]:
from databricks_langchain import VectorSearchRetrieverTool

vs_tool2 = VectorSearchRetrieverTool(
    index_name=f"{UC_CATALOG}.{UC_SCHEMA}.wikipedia_vector_index", # Index name in the format 'catalog.schema.index'
    num_results=5, # Max number of documents to return
    columns=["Genre", "Title", "Plot"], # List of columns to include in the search
    # filters, # Filters to apply to the query
    # query_type, # Query type ("ANN" or "HYBRID").
    tool_name="movie_plots", # Used by the LLM to understand the purpose of the tool
    tool_description="Executes a search on movie plots to find the movies most relevant to the input query.", # Used by the LLM to understand the purpose of the tool
)

vs_tool2.invoke("Jack climbs a beanstalk")


In [0]:
# Bind the retriever tool to your Langchain LLM of choice
llm2 = ChatDatabricks(endpoint=CHAT_MODEL_NAME)
llm_with_tools2 = llm2.bind_tools([vs_tool2])

# Chat with your LLM to test the tool calling functionality
llm_with_tools2.invoke("In which movie does jack climb a beanstalk?")

## Creating a retriever tool as a UC function

In [0]:
%sql
USE hannamoazam_catalog.cookbook;  


CREATE OR REPLACE FUNCTION hannamoazam_catalog.cookbook.wikipedia_vector_search (
  -- The agent uses this comment to determine how to generate the query string parameter.
  query STRING
  COMMENT 'The query string for searching the movie plots.'
) RETURNS TABLE
-- The agent uses this comment to determine when to call this tool. It describes the types of documents and information contained within the index.
COMMENT 'Executes a search on movie plots to find the movies most relevant to the input query.' RETURN
SELECT
  truncated_content as page_content,
  map('Title', Title) as metadata
FROM
  vector_search(
    -- Specify your Vector Search index name here
    index => 'hannamoazam_catalog.cookbook.wikipedia_vector_index',
    query => query,
    num_results => 5
  )
